# Proprietary Momentum Strategy — DJIA 30, Apr 2016 – Apr 2026

**MIT 15.C51 Spring 2026 — Project #1 (Proprietary Trading track, momentum half).**

This notebook develops, backtests, iteratively improves, and documents a proprietary cross-sectional momentum strategy on the Dow Jones 30, culminating in an Investment Committee Memorandum (Stage 7). Mean reversion is a separate deliverable and is explicitly out of scope.

---

### How to run

1. **Kernel → Restart & Run All.** Every cell is designed to execute top-to-bottom from a fresh kernel.
2. Dependencies are installed inline in the first code cell. Required libraries: `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `yfinance`, `pandas_datareader`, `hmmlearn`, `statsmodels`, `xgboost`.
3. Data is fetched live (no cached files committed). Prices/volume via `yfinance`, risk-free rate via FRED `DTB3`, Fama-French factors via `pandas_datareader.famafrench`.
4. All tunable parameters live in the `RUN_CONFIG` dict in Stage 0 — do **not** introduce magic numbers elsewhere. Strategy functions read from `RUN_CONFIG` (passed as `cfg`).
5. Intermediate results are checkpointed to `results.pkl` after Stage 3 and Stage 5 so re-running is cheap.
6. Stage structure follows PROMPT.md §3 — each stage ends with a git commit tagged `stage-N: <what>`.

### Non-negotiables (baked into the pipeline)

- **No look-ahead.** Weights are always shifted by one trading day before being multiplied by realised returns.
- **Point-in-time universe.** The DJIA constituent map is reconstructed from `BASELINE_APR2016` + `CHANGES`; no survivorship bias.
- **Transaction costs.** Applied on portfolio turnover (`weights.diff().abs().sum(axis=1)`) at 10 bps base case. Sensitivities at 0/5/15/20 bps in Stage 6.
- **Walk-forward validation.** Expanding-window folds (see `RUN_CONFIG['WALK_FORWARD_FOLDS']`). No hyperparameter tuning on test windows.
- **Benchmarks.** Every strategy is compared against EW-DJIA buy-and-hold, `^DJI` (price-weighted), SPY, and compounded DTB3.

In [ ]:
# One-shot dependency install. Safe to re-run; pip is idempotent.
# hmmlearn / statsmodels / xgboost / pandas_datareader are not in Colab's default image.
!pip install --quiet hmmlearn statsmodels xgboost pandas_datareader

---

## Stage 0 — Setup & Reproducibility

Single source of truth for every parameter (`RUN_CONFIG`), pinned RNG seeds, version pins for the audit trail, and a global matplotlib style for publication-quality figures.

In [ ]:
from __future__ import annotations

import logging
import random
import sys
import warnings
from dataclasses import dataclass
from typing import Any

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('momentum')

In [ ]:
# ── RUN_CONFIG ────────────────────────────────────────────────────────────
# Single source of truth. Every downstream cell reads from this dict; no
# magic numbers elsewhere in the notebook.

SEED = 42

RUN_CONFIG: dict[str, Any] = {
    # Universe / horizon
    'START_DATE':        '2016-04-18',
    'END_DATE':          '2026-04-18',
    'FFILL_LIMIT_DAYS':  10,

    # Portfolio construction
    'REBALANCE_FREQ':    'ME',           # pandas month-end alias
    'LONG_PCT':          0.20,           # long top quintile
    'SHORT_PCT':         0.20,           # short bottom quintile

    # Transaction costs (per unit turnover, one-way → charged on abs Δweight)
    'COST_BPS':          10,             # base case
    'COST_BPS_GRID':     [0, 5, 10, 15, 20],  # Stage 6 sensitivity

    # Signal lookbacks (trading days)
    'LOOKBACKS': {
        'mom_long':      252,            # 12-month total window
        'mom_mid':       126,            # 6-month
        'mom_short':     63,             # 3-month
        'skip':          21,             # skip-one-month lag
        'vol_long':      60,             # vol estimator
        'vol_short':     20,             # short-horizon vol for scaling
        'ma_fast':       50,
        'ma_slow':       200,
        'beta_window':   252,            # residual-momentum regression window
        'vol_target':    0.10,           # annualised target for Stage 5 vol-targeting
    },

    # Walk-forward expanding-window folds (train_end is inclusive, test_start exclusive of train_end)
    'WALK_FORWARD_FOLDS': [
        {'train_start': '2016-04-18', 'train_end': '2020-12-31',
         'test_start':  '2021-01-01', 'test_end':  '2021-12-31'},
        {'train_start': '2016-04-18', 'train_end': '2021-12-31',
         'test_start':  '2022-01-01', 'test_end':  '2022-12-31'},
        {'train_start': '2016-04-18', 'train_end': '2022-12-31',
         'test_start':  '2023-01-01', 'test_end':  '2026-04-18'},
    ],

    # HMM regime model (Stage 2 strategy #13)
    'HMM_N_STATES':      3,
    'HMM_EXPOSURE':      {'bull': 1.0, 'choppy': 0.5, 'bear': 0.0},

    # Evaluation
    'BOOTSTRAP_N':       1000,
    'BOOTSTRAP_BLOCK':   20,             # block length for stationary bootstrap
    'NEWEY_WEST_LAG':    5,

    # Reproducibility
    'SEED':              SEED,
}

random.seed(SEED)
np.random.seed(SEED)

log.info('RUN_CONFIG loaded: %d top-level keys, seed=%d',
         len(RUN_CONFIG), RUN_CONFIG['SEED'])

In [ ]:
# ── Version pins (audit trail) ────────────────────────────────────────────
# Printed once so the exact environment is recoverable from a saved run.

import importlib

_LIBS = [
    'pandas', 'numpy', 'sklearn', 'matplotlib', 'yfinance',
    'pandas_datareader', 'hmmlearn', 'statsmodels', 'xgboost', 'scipy',
]

print(f'Python           {sys.version.split()[0]}')
for name in _LIBS:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'{name:<18} {ver}')
    except ImportError:
        print(f'{name:<18} NOT INSTALLED')

In [ ]:
# ── Global matplotlib style (set once, inherited by every figure) ─────────

mpl.rcParams.update({
    'figure.figsize':       (12, 5),
    'figure.dpi':           110,
    'savefig.dpi':          160,
    'savefig.bbox':         'tight',
    'font.family':          'DejaVu Sans',
    'font.size':            10,
    'axes.titlesize':       12,
    'axes.titleweight':     'semibold',
    'axes.labelsize':       10,
    'axes.grid':            True,
    'axes.spines.top':      False,
    'axes.spines.right':    False,
    'axes.prop_cycle':      mpl.cycler(color=[
        '#0B3D91',   # deep navy
        '#C0392B',   # brick
        '#117A65',   # forest
        '#B9770E',   # burnt orange
        '#6C3483',   # plum
        '#7F8C8D',   # slate
    ]),
    'grid.alpha':           0.25,
    'grid.linestyle':       '--',
    'lines.linewidth':      1.6,
    'legend.frameon':       False,
    'legend.fontsize':      9,
})

log.info('matplotlib style configured')

---

## Stage 1 — Data Layer

_Point-in-time DJIA constituents, yfinance price/volume pull, FRED DTB3 risk-free rate, universe validation visuals. Implemented in the next stage commit._

---

## Stage 2 — Strategy Zoo (15 strategies)

_Classical, advanced, and ML-stacked momentum variants. Each implemented as a pure function `strategy_name(prices, volume, constituent_map, cfg) -> weights_df`. Implemented in the next stage commit._

---

## Stage 3 — Evaluation Framework & Leaderboard

_Single `evaluate()` producing risk-adjusted metrics, FF3 regression, bootstrap Sharpe CI, Newey-West t-stats. Master leaderboard sorted by walk-forward OOS Sharpe. Implemented in the next stage commit._

---

## Stage 4 — Diagnostic Visuals (Top 5)

_Cumulative curves, rolling Sharpe, underwater, return distribution, monthly heatmap, FF3 exposure, turnover. Top 5 by OOS Sharpe only. Implemented in the next stage commit._

---

## Stage 5 — Iterative Improvement Loop

_Up to five rounds of diagnose → propose → implement → compare → decide, starting from the #1 ranked strategy. Early-stop when OOS Sharpe gain < 0.05. Implemented in the next stage commit._

---

## Stage 6 — Robustness Tests (Winner)

_Cost/frequency/lookback sensitivities, subperiod stability, Monte Carlo bootstrap, stress-period P&L, capacity estimate. Implemented in the next stage commit._

---

## Stage 7 — Investment Committee Memorandum

_Executive summary, strategy rationale, P&L conditions, statistical properties, 10-year performance review, risk factors, recommendation, Appendix A (source attribution incl. verbatim PROMPT.md and `llm_interactions.log`), Appendix B (iteration log). Written in the next stage commit._